In [1]:
# ================================================================
# INFERENCE PIPELINE — Horizon-Aware Quantile Regressor (single model)
#   - Fully aligned with updated training pipeline (raw sliders, INTs off)
#   - Artifacts suffixed _omicron
#   - (NEW) lag gating at serve time: lag_gate=0 and all lag features zeroed in Z-space
# ================================================================

import os, json, math
import numpy as np
import pandas as pd
import torch
from google.colab import drive

# ------------------- CONFIG -------------------
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

drive.mount("/content/drive")

# Default paths (override if needed)
DATA_PATH = "/content/drive/MyDrive/Data/df_final_omicron.csv"
MODEL_DIR = "/content/drive/MyDrive/Models"

HALFLIVES_POLICY = (6, 12)
EWM_TAGS         = [f"ewm_hl{hl}" for hl in HALFLIVES_POLICY]

# Discrete policy lags used in training
LAG_STEPS = (1, 2, 3, 4)

# Raw policy slider columns used in training/inference
POLICY_COLS = (
    "covid_19_policy_stringency",
    "covid_19_face_covering_policy",
    "covid_19_testing_tracing_policy",
)

# (Training has interactions OFF; keep helper for compatibility)
INT_MOD_RELU_NAMES = tuple()  # no *_relu interactions active

H = 12  # prediction horizon per training export

REQUIRED_ARTIFACTS = dict(
    SCRIPTED_PT       = "model_scripted_omicron.pt",       # scripted wrapper applies sigmoid & scales by U
    FEATURE_CONTRACT  = "feature_contract_omicron.json",   # has `feature_order`
    SCHEMA_JSON       = "serving_schema_omicron.json",
    FEATURE_NORM_JSON = "feature_norm_stats_omicron.json", # mean/std per feature (KNOWN_COLS)
    CENTER_JSON       = "center_means_omicron.json",       # train means for mains/moderators (for *_c)
)

# ------------------- SMART ARTIFACT LOADER -------------------
def _load_json(p):
    with open(p, "r") as f:
        return json.load(f)

def _first_dir_with_all(files_map, dirs):
    best_d, best_count = None, -1
    for d in dirs:
        missing = [fname for fname in files_map.values() if not os.path.exists(os.path.join(d, fname))]
        if not missing:
            return d, []
        count = len(files_map) - len(missing)
        if count > best_count:
            best_d, best_count = d, count
    missing = [fname for fname in files_map.values() if not os.path.exists(os.path.join(best_d, fname))]
    return best_d, missing

CANDIDATE_DIRS = [
    MODEL_DIR,
    "/content/drive/MyDrive/Models",
    "./models",
    "./",
    "../models",
]

MODEL_DIR, _missing = _first_dir_with_all(REQUIRED_ARTIFACTS, CANDIDATE_DIRS)
if _missing:
    print("Artifact discovery summary:")
    for cand in CANDIDATE_DIRS:
        hits = [k for k,v in REQUIRED_ARTIFACTS.items() if os.path.exists(os.path.join(cand, v))]
        print(f" - {cand}: found {len(hits)}/{len(REQUIRED_ARTIFACTS)} -> {hits}")
    raise FileNotFoundError(
        f"Could not find all required artifacts in any of {CANDIDATE_DIRS}.\n"
        f"Best candidate: {MODEL_DIR}\n"
        f"Missing there: {_missing}\n"
        f"Tip: set MODEL_DIR to the folder you used during training/export."
    )

# Load artifacts
art = {k: os.path.join(MODEL_DIR, v) for k, v in REQUIRED_ARTIFACTS.items() }
feature_contract = _load_json(art["FEATURE_CONTRACT"])
schema           = _load_json(art["SCHEMA_JSON"])
feature_norm     = _load_json(art["FEATURE_NORM_JSON"])
center_means     = _load_json(art["CENTER_JSON"])

# Sanity: feature order matches KNOWN_COLS used at training (which includes 'lag_gate' at the end)
feature_order = feature_contract.get("feature_order", [])
if not feature_order or not isinstance(feature_order, list):
    raise ValueError("feature_contract.feature_order missing or invalid.")
D_known = int(feature_contract.get("decoder_cont_lastdim", len(feature_order)))
assert D_known == len(feature_order), "Feature contract dimension mismatch."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = torch.jit.load(art["SCRIPTED_PT"], map_location=device).eval()
print(f"✅ Loaded artifacts from: {MODEL_DIR}")

# ------------------- DATA (history) -------------------
df = pd.read_csv(DATA_PATH)
df = df.sort_values(["country_iso3", "week_id"]).reset_index(drop=True)

# ------------------- HELPERS -------------------
def _ewm_alpha(halflife: float) -> float:
    return 1.0 - math.exp(-math.log(2.0) / float(halflife))

def _series_ewm(series, hl):
    """Causal EWM (Koyck) for a 1D arraylike (aligns to pandas ewm(adjust=False))."""
    a = _ewm_alpha(hl)
    out = np.empty(len(series), dtype=float)
    prev = None
    for i, xi in enumerate(np.asarray(series, dtype=float)):
        prev = xi if i == 0 else (a*float(xi) + (1.0 - a)*float(prev))
        out[i] = prev
    return out

def _resolve_week_id_for_country(df_country, week_id_or_idx):
    """Accept true week_id or 0-based positional index into that country's history."""
    wk = int(week_id_or_idx)
    wkvals = df_country["week_id"].to_numpy()
    if (wk == wkvals).any():    # it's a true week_id
        return wk
    if 0 <= wk < len(wkvals):   # positional index
        return int(wkvals[wk])
    raise ValueError(f"{wk} not valid week_id or positional index (0..{len(wkvals)-1}).")

def _normalize_scalar(name, raw_value, norm_stats):
    mu = float(norm_stats[name]["mean"])
    sd = float(norm_stats[name]["std"]) or 1.0
    return (float(raw_value) - mu) / sd

def _policy_base_names():
    """List of policy Koyck window column names (raw policy sliders; no PCA)."""
    return [f"{feat}_{tag}" for tag in EWM_TAGS for feat in POLICY_COLS]

def _future_slider_path_from_data(g, start_week_id, horizon):
    """
    Baseline path from observed data: take sliders (S, M, T) for weeks (t0+1..t0+H); pad with last if shorter.
    """
    fut = g[(g["week_id"] > start_week_id) & (g["week_id"] <= start_week_id + horizon)].copy()
    triples = fut[list(POLICY_COLS)].astype(float).to_numpy()
    if len(triples) == 0:
        # No observed future — repeat t0 triple
        row0 = g[g["week_id"] == start_week_id].iloc[0]
        triples = np.repeat(np.array([
            float(row0[POLICY_COLS[0]]),
            float(row0[POLICY_COLS[1]]),
            float(row0[POLICY_COLS[2]]),
        ])[None, :], horizon, axis=0)
        return triples
    if len(triples) < horizon:
        pad = np.repeat(triples[-1][None, :], horizon - len(triples), axis=0)
        triples = np.vstack([triples, pad])
    return triples

def _build_future_policy_windows(history_df, start_week_id, future_sliders, horizon=H):
    """
    Build step-varying Koyck-smoothed RAW policy windows from history + future slider path.
    future_sliders: (3,) or (H,3) of raw levels (S, M, T).
    Returns dict: {f"{raw_name}_ewm_hl{6|12}"} → np.ndarray length H
    """
    g = history_df[history_df["week_id"] <= int(start_week_id)].copy().sort_values("week_id")
    if g.empty:
        raise ValueError("No history available to build Koyck windows.")

    # history EWM states
    last_ewm = {}
    for raw in POLICY_COLS:
        hist_vals = g[raw].astype(float).to_numpy()
        for hl in HALFLIVES_POLICY:
            sm = _series_ewm(hist_vals, hl)
            last_ewm[(raw, hl)] = float(sm[-1])

    fs = np.asarray(future_sliders, dtype=float)
    if fs.ndim == 1:
        assert fs.shape[0] == 3, "Single future_sliders triple must be length 3."
        future_triples = np.repeat(fs[None, :], horizon, axis=0)
    else:
        assert fs.shape[1] == 3 and fs.shape[0] >= horizon, "future_sliders must be (H,3) or a single length-3 triple."
        future_triples = fs

    out = {f"{raw}_ewm_hl{hl}": np.empty(horizon, dtype=float)
           for raw in POLICY_COLS for hl in HALFLIVES_POLICY}

    for h in range(horizon):
        s_raw, m_raw, t_raw = [float(x) for x in future_triples[h]]
        raw_vals = {
            "covid_19_policy_stringency": s_raw,
            "covid_19_face_covering_policy": m_raw,
            "covid_19_testing_tracing_policy": t_raw,
        }
        for raw in POLICY_COLS:
            xh_raw = raw_vals[raw]
            for hl in HALFLIVES_POLICY:
                a = _ewm_alpha(hl)
                prev = last_ewm[(raw, hl)]
                newv = a * xh_raw + (1.0 - a) * prev
                last_ewm[(raw, hl)] = newv
                out[f"{raw}_ewm_hl{hl}"][h] = newv
    return out

# ------------------- Build discrete policy lags over horizon -------------------
def _build_future_policy_lags(history_df, start_week_id, future_sliders, horizon=H, lag_steps=LAG_STEPS):
    """
    Build simple discrete lags of RAW policy sliders from history + future path.
    Uses the training convention: first k weeks fall back to the *current* value (shift(k).fillna(current)).
    Returns dict: {f"{raw}_lag{k}"} -> np.ndarray length H
    """
    g = history_df[history_df["week_id"] <= int(start_week_id)].copy().sort_values("week_id")
    if g.empty:
        raise ValueError("No history available to build policy lags.")

    fs = np.asarray(future_sliders, dtype=float)
    if fs.ndim == 1:
        assert fs.shape[0] == 3, "Single future_sliders triple must be length 3."
        future_triples = np.repeat(fs[None, :], horizon, axis=0)
    else:
        assert fs.shape[1] == 3 and fs.shape[0] >= horizon, "future_sliders must be (H,3) or a single length-3 triple."
        future_triples = fs

    # build per-policy sequences: history (<= t0) + future (t0+1..t0+H)
    hist = {p: g[p].astype(float).to_numpy() for p in POLICY_COLS}
    fut_map = {
        "covid_19_policy_stringency": future_triples[:, 0].astype(float),
        "covid_19_face_covering_policy": future_triples[:, 1].astype(float),
        "covid_19_testing_tracing_policy": future_triples[:, 2].astype(float),
    }

    out = {f"{p}_lag{k}": np.empty(horizon, dtype=float)
           for p in POLICY_COLS for k in lag_steps}

    for p in POLICY_COLS:
        seq = np.concatenate([hist[p], fut_map[p]], axis=0)  # length = len(hist) + H
        T_hist = len(hist[p])
        # step h corresponds to index T_hist + (h-1)
        for h in range(1, horizon+1):
            cur_idx = T_hist + (h - 1)
            for k in lag_steps:
                lag_idx = cur_idx - k
                if lag_idx < 0:
                    # align with training's .shift(k).fillna(current)
                    val = seq[cur_idx]
                else:
                    val = seq[lag_idx]
                out[f"{p}_lag{k}"][h-1] = float(val)
    return out

def _extract_knowns_fixed(df_country, start_week_id, horizon):
    """
    FIXED OVER H (aligns with training): take t0 seasonality & latitude and hold constant.
    """
    row0 = df_country[df_country["week_id"] == int(start_week_id)]
    if row0.empty:
        raise ValueError(f"week_id={start_week_id} not found.")
    row0 = row0.iloc[0].to_dict()
    latitude = float(row0.get("latitude", 0.0))
    sine0    = float(row0.get("sine_seasonality", 0.0))
    cosine0  = float(row0.get("cosine_seasonality", 1.0))
    rows = []
    for h in range(1, horizon + 1):
        rows.append(dict(
            week_id=int(start_week_id) + h,
            sine_seasonality=sine0,
            cosine_seasonality=cosine0,
            latitude=latitude,
        ))
    return pd.DataFrame(rows)

def _compute_prev_anchors_at_t0(df_country, week_id, trend_window=4):
    """
    Compute prev_ma_4w, prev_slope_4w at anchor time (using history up to week_id, inclusive).
    """
    g = df_country[df_country["week_id"] <= int(week_id)].copy()
    y = g["covid_19_prevalence"].astype(float).to_numpy()
    ma = float(np.mean(y[-trend_window:])) if y.size >= 1 else 0.0
    yy = y[-trend_window:]
    k = len(yy)
    if k < 2:
        slope = 0.0
    else:
        x = np.arange(k, dtype=float)
        xm, ym = x.mean(), yy.mean()
        denom = np.sum((x - xm)**2)
        slope = float(np.sum((x - xm)*(yy - ym)) / denom) if denom > 0 else 0.0
    return ma, slope

def _recompute_interactions_inplace(step_raw_features, norm_stats, policy_base_names):
    """
    (Aligned with training) — interactions are OFF.
    Function kept for API compatibility; does nothing.
    """
    return

# ------------------- MAIN PREDICTOR -------------------
def predict_horizon(country_iso3, week_id_or_idx, policy_sliders=(0.0, 0.0, 0.0)):
    """
    country_iso3: string (e.g., "USA")
    week_id_or_idx: int (either actual week_id or 0-based index)
    policy_sliders: (3,) or (H,3) = (stringency, face coverings, testing/tracing) raw levels
    Returns:
      DataFrame with columns:
        country_iso3, week_id, q10, q50, q90, dq10, dq50, dq90
      where dq* are deltas relative to the first horizon step (h=1).
    """
    country = str(country_iso3)
    g = df[df["country_iso3"] == country].sort_values("week_id").reset_index(drop=True)
    if g.empty:
        raise ValueError(f"No data for {country}")

    # Resolve anchor week_id
    week_id = _resolve_week_id_for_country(g, week_id_or_idx)

    # ---- constants at anchor (kept across horizon) ----
    r0 = g.loc[g["week_id"] == week_id].iloc[0]

    # centered mains (time_c uses train center "time" which equals week_id in training)
    time_center = float(center_means.get("time", 0.0))
    time_c = float(week_id) - time_center

    # Known controls per step (seasonality FIXED across H; latitude constant)
    fut_ctrl = _extract_knowns_fixed(g, week_id, H)
    latitude = float(fut_ctrl.loc[0, "latitude"])
    sine0    = float(fut_ctrl.loc[0, "sine_seasonality"])
    cosine0  = float(fut_ctrl.loc[0, "cosine_seasonality"])

    # Anchors from history at t0
    prev_ma, prev_slope = _compute_prev_anchors_at_t0(g, week_id, trend_window=4)
    prev_ma_c    = prev_ma    - float(center_means.get("prev_ma_4w", prev_ma))
    prev_slope_c = prev_slope - float(center_means.get("prev_slope_4w", prev_slope))

    # Policy windows (RAW) & policy lags (RAW) from history + future path
    windows = _build_future_policy_windows(g, week_id, policy_sliders, H)
    lags    = _build_future_policy_lags(g, week_id, policy_sliders, H, LAG_STEPS)

    # Build raw → normalized features per step (and interactions)
    policy_base_names = _policy_base_names()
    step_rows = []
    for h in range(H):
        # RAW values first (for correct normalization)
        sr = {
            # ---- anchor mains (centered where *_c) ----
            "prev_ma_4w_c": prev_ma_c,
            "prev_slope_4w_c": prev_slope_c,
            "time_c": time_c,

            # step-known controls (seasonality & × latitude FIXED; latitude constant)
            "latitude": latitude,
            "sine_seasonality": sine0,
            "cosine_seasonality": cosine0,
            "sine_seasonality_x_latitude": sine0 * latitude,
            "cosine_seasonality_x_latitude": cosine0 * latitude,
        }

        # ------------------ (A) GATE: add lag_gate raw (will be z-scored) ------------------
        sr["lag_gate"] = 0.0  # serve with gate OFF

        # ------------------ (B) policy Koyck windows ------------------
        for name in policy_base_names:
            sr[name] = float(windows[name][h])

        # ------------------ (C) discrete lag features (RAW) ------------------
        for p in POLICY_COLS:
            for k in LAG_STEPS:
                lname = f"{p}_lag{k}"
                sr[lname] = float(lags[lname][h]) if lname in lags else feature_norm[lname]["mean"]

        # --- Z-score everything present ---
        z = {k: _normalize_scalar(k, v, feature_norm) for k, v in sr.items() if k in feature_norm}

        # ------------------ (D) GATE in Z-space: zero all lag features ------------------
        for p in POLICY_COLS:
            for k in LAG_STEPS:
                lname = f"{p}_lag{k}"
                if lname in z:
                    z[lname] = 0.0  # hard zero in Z-space when gate=0

        # (Interactions are OFF; keep no-op for compatibility)
        _recompute_interactions_inplace(sr, feature_norm, policy_base_names)

        # Ensure any (future) normalized interaction features present (no-ops here)
        for k in feature_order:
            if "_x_" in k and k not in z and k in sr:
                z[k] = float(sr[k])

        # Order to match training
        step_rows.append([z.get(n, 0.0) for n in feature_order])

    # [B=1, H, D]
    decoder_cont = torch.tensor(step_rows, dtype=torch.float32, device=device).unsqueeze(0)

    # Scripted model returns already scaled to [0, U] (U=1.0 per export unless changed)
    with torch.no_grad():
        y = model(decoder_cont).squeeze(0).cpu().numpy()  # [H, 3]

    weeks = [int(week_id) + h for h in range(1, H + 1)]
    out = pd.DataFrame({
        "country_iso3": country, "week_id": weeks,
        "q10": y[:, 0], "q50": y[:, 1], "q90": y[:, 2]
    })

    # === Δ over horizon relative to first step (h=1) ===
    base_q = out.loc[out.index.min(), ["q10","q50","q90"]].to_numpy(dtype=float)
    deltas = out[["q10","q50","q90"]].to_numpy(dtype=float) - base_q
    out["dq10"] = deltas[:, 0]
    out["dq50"] = deltas[:, 1]
    out["dq90"] = deltas[:, 2]

    return out

# ------------------- Convenience: inspect a country's week indices -------------------
def describe_country_weeks(country_iso3, n_head=5, n_tail=5):
    g = df[df["country_iso3"] == str(country_iso3)].sort_values("week_id").reset_index(drop=True)
    print(f"{country_iso3}: {len(g)} rows (positional indices 0..{len(g)-1})")
    try:
        from IPython.display import display
        display(g.head(n_head)[["week_id", "year", "iso_week"]])
        display(g.tail(n_tail)[["week_id", "year", "iso_week"]])
    except Exception:
        print(g.head(n_head)[["week_id", "year", "iso_week"]])
        print(g.tail(n_tail)[["week_id", "year", "iso_week"]])

# ------------------- Scenario comparison helper -------------------
def compare_horizons(country_iso3, week_id_or_idx, scenario_sliders):
    """
    Return only q50_baseline, q50_scenario, pct_change_q50.
    Baseline = observed policy path; Scenario = user-specified sliders.
    """
    country = str(country_iso3)
    g = df[df["country_iso3"] == country].sort_values("week_id").reset_index(drop=True)
    if g.empty:
        raise ValueError(f"No data for {country}")
    week_id = _resolve_week_id_for_country(g, week_id_or_idx)

    # Observed baseline policy path (t0+1..t0+H)
    baseline_path = _future_slider_path_from_data(g, week_id, H)

    preds_baseline = predict_horizon(country_iso3, week_id, baseline_path)
    preds_scenario = predict_horizon(country_iso3, week_id, scenario_sliders)

    df_cmp = preds_baseline.merge(
        preds_scenario, on=["country_iso3","week_id"], suffixes=("_baseline", "_scenario")
    )

    # % change in the q50 level (scenario vs baseline)
    denom = df_cmp["q50_baseline"].replace({0.0: np.nan})
    df_cmp["pct_change_q50"] = 100.0 * (df_cmp["q50_scenario"] - df_cmp["q50_baseline"]) / denom

    # Return only the requested columns (keep identifiers for clarity)
    return df_cmp[["country_iso3", "week_id", "q50_baseline", "q50_scenario", "pct_change_q50"]]


# ------------------- Example usage -------------------
# describe_country_weeks("USA")
# preds = predict_horizon("USA", 120, (40.0, 50.0, 60.0))   # raw levels
# from IPython.display import display
# display(preds.head(12))   # includes dq10/dq50/dq90 relative to h=1
#
# cmp = compare_horizons("AUT", 91, (0.0, 0.0, 0.0))
# display(cmp.head(12))     # includes pct_change_q50


Mounted at /content/drive
✅ Loaded artifacts from: /content/drive/MyDrive/Models


In [4]:
# ================================================================
# INFERENCE PIPELINE — Horizon-Aware Quantile Regressor (single model)
#   - Fully aligned with updated training pipeline (raw sliders, INTs off)
#   - Artifacts suffixed _omicron
#   - lag gating at serve time: lag_gate=0 and all lag features zeroed in Z-space
#   - (NEW) calendar-driven, step-varying seasonality over the horizon
# ================================================================

import os, json, math
import numpy as np
import pandas as pd
import torch
from google.colab import drive

# ------------------- CONFIG -------------------
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

drive.mount("/content/drive")

# Default paths (override if needed)
DATA_PATH = "/content/drive/MyDrive/Data/df_final_omicron.csv"
MODEL_DIR = "/content/drive/MyDrive/Models"

HALFLIVES_POLICY = (6, 12)
EWM_TAGS         = [f"ewm_hl{hl}" for hl in HALFLIVES_POLICY]

# Discrete policy lags used in training
LAG_STEPS = (1, 2, 3, 4)

# Raw policy slider columns used in training/inference
POLICY_COLS = (
    "covid_19_policy_stringency",
    "covid_19_face_covering_policy",
    "covid_19_testing_tracing_policy",
)

# (Training has interactions OFF; keep helper for compatibility)
INT_MOD_RELU_NAMES = tuple()  # no *_relu interactions active

H = 12  # prediction horizon per training export

REQUIRED_ARTIFACTS = dict(
    SCRIPTED_PT       = "model_scripted_omicron.pt",       # scripted wrapper applies sigmoid & scales by U
    FEATURE_CONTRACT  = "feature_contract_omicron.json",   # has `feature_order`
    SCHEMA_JSON       = "serving_schema_omicron.json",
    FEATURE_NORM_JSON = "feature_norm_stats_omicron.json", # mean/std per feature (KNOWN_COLS)
    CENTER_JSON       = "center_means_omicron.json",       # train means for mains/moderators (for *_c)
)

# ------------------- SMART ARTIFACT LOADER -------------------
def _load_json(p):
    with open(p, "r") as f:
        return json.load(f)

def _first_dir_with_all(files_map, dirs):
    best_d, best_count = None, -1
    for d in dirs:
        missing = [fname for fname in files_map.values() if not os.path.exists(os.path.join(d, fname))]
        if not missing:
            return d, []
        count = len(files_map) - len(missing)
        if count > best_count:
            best_d, best_count = d, count
    missing = [fname for fname in files_map.values() if not os.path.exists(os.path.join(best_d, fname))]
    return best_d, missing

CANDIDATE_DIRS = [
    MODEL_DIR,
    "/content/drive/MyDrive/Models",
    "./models",
    "./",
    "../models",
]

MODEL_DIR, _missing = _first_dir_with_all(REQUIRED_ARTIFACTS, CANDIDATE_DIRS)
if _missing:
    print("Artifact discovery summary:")
    for cand in CANDIDATE_DIRS:
        hits = [k for k,v in REQUIRED_ARTIFACTS.items() if os.path.exists(os.path.join(cand, v))]
        print(f" - {cand}: found {len(hits)}/{len(REQUIRED_ARTIFACTS)} -> {hits}")
    raise FileNotFoundError(
        f"Could not find all required artifacts in any of {CANDIDATE_DIRS}.\n"
        f"Best candidate: {MODEL_DIR}\n"
        f"Missing there: {_missing}\n"
        f"Tip: set MODEL_DIR to the folder you used during training/export."
    )

# Load artifacts
art = {k: os.path.join(MODEL_DIR, v) for k, v in REQUIRED_ARTIFACTS.items() }
feature_contract = _load_json(art["FEATURE_CONTRACT"])
schema           = _load_json(art["SCHEMA_JSON"])
feature_norm     = _load_json(art["FEATURE_NORM_JSON"])
center_means     = _load_json(art["CENTER_JSON"])

# Sanity: feature order matches KNOWN_COLS used at training (which includes 'lag_gate' at the end)
feature_order = feature_contract.get("feature_order", [])
if not feature_order or not isinstance(feature_order, list):
    raise ValueError("feature_contract.feature_order missing or invalid.")
D_known = int(feature_contract.get("decoder_cont_lastdim", len(feature_order)))
assert D_known == len(feature_order), "Feature contract dimension mismatch."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = torch.jit.load(art["SCRIPTED_PT"], map_location=device).eval()
print(f"✅ Loaded artifacts from: {MODEL_DIR}")

# ------------------- DATA (history) -------------------
df = pd.read_csv(DATA_PATH)
df = df.sort_values(["country_iso3", "week_id"]).reset_index(drop=True)

# ------------------- HELPERS -------------------
def _ewm_alpha(halflife: float) -> float:
    return 1.0 - math.exp(-math.log(2.0) / float(halflife))

def _series_ewm(series, hl):
    """Causal EWM (Koyck) for a 1D arraylike (aligns to pandas ewm(adjust=False))."""
    a = _ewm_alpha(hl)
    out = np.empty(len(series), dtype=float)
    prev = None
    for i, xi in enumerate(np.asarray(series, dtype=float)):
        prev = xi if i == 0 else (a*float(xi) + (1.0 - a)*float(prev))
        out[i] = prev
    return out

def _resolve_week_id_for_country(df_country, week_id_or_idx):
    """Accept true week_id or 0-based positional index into that country's history."""
    wk = int(week_id_or_idx)
    wkvals = df_country["week_id"].to_numpy()
    if (wk == wkvals).any():    # it's a true week_id
        return wk
    if 0 <= wk < len(wkvals):   # positional index
        return int(wkvals[wk])
    raise ValueError(f"{wk} not valid week_id or positional index (0..{len(wkvals)-1}).")

def _normalize_scalar(name, raw_value, norm_stats):
    mu = float(norm_stats[name]["mean"])
    sd = float(norm_stats[name]["std"]) or 1.0
    return (float(raw_value) - mu) / sd

def _policy_base_names():
    """List of policy Koyck window column names (raw policy sliders; no PCA)."""
    return [f"{feat}_{tag}" for tag in EWM_TAGS for feat in POLICY_COLS]

def _future_slider_path_from_data(g, start_week_id, horizon):
    """
    Baseline path from observed data: take sliders (S, M, T) for weeks (t0+1..t0+H); pad with last if shorter.
    """
    fut = g[(g["week_id"] > start_week_id) & (g["week_id"] <= start_week_id + horizon)].copy()
    triples = fut[list(POLICY_COLS)].astype(float).to_numpy()
    if len(triples) == 0:
        # No observed future — repeat t0 triple
        row0 = g[g["week_id"] == start_week_id].iloc[0]
        triples = np.repeat(np.array([
            float(row0[POLICY_COLS[0]]),
            float(row0[POLICY_COLS[1]]),
            float(row0[POLICY_COLS[2]]),
        ])[None, :], horizon, axis=0)
        return triples
    if len(triples) < horizon:
        pad = np.repeat(triples[-1][None, :], horizon - len(triples), axis=0)
        triples = np.vstack([triples, pad])
    return triples

def _build_future_policy_windows(history_df, start_week_id, future_sliders, horizon=H):
    """
    Build step-varying Koyck-smoothed RAW policy windows from history + future slider path.
    future_sliders: (3,) or (H,3) of raw levels (S, M, T).
    Returns dict: {f"{raw_name}_ewm_hl{6|12}"} → np.ndarray length H
    """
    g = history_df[history_df["week_id"] <= int(start_week_id)].copy().sort_values("week_id")
    if g.empty:
        raise ValueError("No history available to build Koyck windows.")

    # history EWM states
    last_ewm = {}
    for raw in POLICY_COLS:
        hist_vals = g[raw].astype(float).to_numpy()
        for hl in HALFLIVES_POLICY:
            sm = _series_ewm(hist_vals, hl)
            last_ewm[(raw, hl)] = float(sm[-1])

    fs = np.asarray(future_sliders, dtype=float)
    if fs.ndim == 1:
        assert fs.shape[0] == 3, "Single future_sliders triple must be length 3."
        future_triples = np.repeat(fs[None, :], horizon, axis=0)
    else:
        assert fs.shape[1] == 3 and fs.shape[0] >= horizon, "future_sliders must be (H,3) or a single length-3 triple."
        future_triples = fs

    out = {f"{raw}_ewm_hl{hl}": np.empty(horizon, dtype=float)
           for raw in POLICY_COLS for hl in HALFLIVES_POLICY}

    for h in range(horizon):
        s_raw, m_raw, t_raw = [float(x) for x in future_triples[h]]
        raw_vals = {
            "covid_19_policy_stringency": s_raw,
            "covid_19_face_covering_policy": m_raw,
            "covid_19_testing_tracing_policy": t_raw,
        }
        for raw in POLICY_COLS:
            xh_raw = raw_vals[raw]
            for hl in HALFLIVES_POLICY:
                a = _ewm_alpha(hl)
                prev = last_ewm[(raw, hl)]
                newv = a * xh_raw + (1.0 - a) * prev
                last_ewm[(raw, hl)] = newv
                out[f"{raw}_ewm_hl{hl}"][h] = newv
    return out

# ------------------- Build discrete policy lags over horizon -------------------
def _build_future_policy_lags(history_df, start_week_id, future_sliders, horizon=H, lag_steps=LAG_STEPS):
    """
    Build simple discrete lags of RAW policy sliders from history + future path.
    Uses the training convention: first k weeks fall back to the *current* value (shift(k).fillna(current)).
    Returns dict: {f"{raw}_lag{k}"} -> np.ndarray length H
    """
    g = history_df[history_df["week_id"] <= int(start_week_id)].copy().sort_values("week_id")
    if g.empty:
        raise ValueError("No history available to build policy lags.")

    fs = np.asarray(future_sliders, dtype=float)
    if fs.ndim == 1:
        assert fs.shape[0] == 3, "Single future_sliders triple must be length 3."
        future_triples = np.repeat(fs[None, :], horizon, axis=0)
    else:
        assert fs.shape[1] == 3 and fs.shape[0] >= horizon, "future_sliders must be (H,3) or a single length-3 triple."
        future_triples = fs

    # build per-policy sequences: history (<= t0) + future (t0+1..t0+H)
    hist = {p: g[p].astype(float).to_numpy() for p in POLICY_COLS}
    fut_map = {
        "covid_19_policy_stringency": future_triples[:, 0].astype(float),
        "covid_19_face_covering_policy": future_triples[:, 1].astype(float),
        "covid_19_testing_tracing_policy": future_triples[:, 2].astype(float),
    }

    out = {f"{p}_lag{k}": np.empty(horizon, dtype=float)
           for p in POLICY_COLS for k in lag_steps}

    for p in POLICY_COLS:
        seq = np.concatenate([hist[p], fut_map[p]], axis=0)  # length = len(hist) + H
        T_hist = len(hist[p])
        # step h corresponds to index T_hist + (h-1)
        for h in range(1, horizon+1):
            cur_idx = T_hist + (h - 1)
            for k in lag_steps:
                lag_idx = cur_idx - k
                if lag_idx < 0:
                    # align with training's .shift(k).fillna(current)
                    val = seq[cur_idx]
                else:
                    val = seq[lag_idx]
                out[f"{p}_lag{k}"][h-1] = float(val)
    return out

# --- Step-varying seasonality helpers (deterministic by calendar) ---
WEEKS_PER_YEAR = 52.1775  # keep consistent with training

def _seasonal_for_week(week_id: int, latitude: float):
    angle = 2.0 * math.pi * (week_id / WEEKS_PER_YEAR)
    s = math.sin(angle)
    c = math.cos(angle)
    return {
        "sine_seasonality": s,
        "cosine_seasonality": c,
        "sine_seasonality_x_latitude": s * float(latitude),
        "cosine_seasonality_x_latitude": c * float(latitude),
    }

def _extract_knowns_rolling(df_country, start_week_id, horizon):
    """
    Seasonality varies with h (week_id+t); latitude is held constant.
    """
    row0 = df_country[df_country["week_id"] == int(start_week_id)]
    if row0.empty:
        raise ValueError(f"week_id={start_week_id} not found.")
    lat = float(row0.iloc[0]["latitude"])

    rows = []
    for h in range(1, horizon + 1):
        wk = int(start_week_id) + h
        seas = _seasonal_for_week(wk, lat)
        rows.append(dict(week_id=wk, latitude=lat, **seas))
    return pd.DataFrame(rows)

def _compute_prev_anchors_at_t0(df_country, week_id, trend_window=4):
    """
    Compute prev_ma_4w, prev_slope_4w at anchor time (using history up to week_id, inclusive).
    """
    g = df_country[df_country["week_id"] <= int(week_id)].copy()
    y = g["covid_19_prevalence"].astype(float).to_numpy()
    ma = float(np.mean(y[-trend_window:])) if y.size >= 1 else 0.0
    yy = y[-trend_window:]
    k = len(yy)
    if k < 2:
        slope = 0.0
    else:
        x = np.arange(k, dtype=float)
        xm, ym = x.mean(), yy.mean()
        denom = np.sum((x - xm)**2)
        slope = float(np.sum((x - xm)*(yy - ym)) / denom) if denom > 0 else 0.0
    return ma, slope

def _recompute_interactions_inplace(step_raw_features, norm_stats, policy_base_names):
    """
    (Aligned with current config) — interactions are OFF.
    Function kept for API compatibility; does nothing.
    """
    return

# ------------------- MAIN PREDICTOR -------------------
def predict_horizon(country_iso3, week_id_or_idx, policy_sliders=(0.0, 0.0, 0.0)):
    """
    country_iso3: string (e.g., "USA")
    week_id_or_idx: int (either actual week_id or 0-based index)
    policy_sliders: (3,) or (H,3) = (stringency, face coverings, testing/tracing) raw levels
    Returns:
      DataFrame with columns:
        country_iso3, week_id, q10, q50, q90, dq10, dq50, dq90
      where dq* are deltas relative to the first horizon step (h=1).
    """
    country = str(country_iso3)
    g = df[df["country_iso3"] == country].sort_values("week_id").reset_index(drop=True)
    if g.empty:
        raise ValueError(f"No data for {country}")

    # Resolve anchor week_id
    week_id = _resolve_week_id_for_country(g, week_id_or_idx)

    # ---- constants at anchor (kept across horizon) ----
    r0 = g.loc[g["week_id"] == week_id].iloc[0]

    # centered mains (time_c uses train center "time" which equals week_id in training)
    time_center = float(center_means.get("time", 0.0))
    time_c = float(week_id) - time_center

    # Known controls per step (seasonality ROLLS across H; latitude constant)
    fut_ctrl = _extract_knowns_rolling(g, week_id, H)

    # Anchors from history at t0
    prev_ma, prev_slope = _compute_prev_anchors_at_t0(g, week_id, trend_window=4)
    prev_ma_c    = prev_ma    - float(center_means.get("prev_ma_4w", prev_ma))
    prev_slope_c = prev_slope - float(center_means.get("prev_slope_4w", prev_slope))

    # Policy windows (RAW) & policy lags (RAW) from history + future path
    windows = _build_future_policy_windows(g, week_id, policy_sliders, H)
    lags    = _build_future_policy_lags(g, week_id, policy_sliders, H, LAG_STEPS)

    # Build raw → normalized features per step (and interactions)
    policy_base_names = _policy_base_names()
    step_rows = []
    for h in range(H):
        wkrow = fut_ctrl.iloc[h]

        # RAW values first (for correct normalization)
        sr = {
            # ---- anchor mains (centered where *_c) ----
            "prev_ma_4w_c": prev_ma_c,
            "prev_slope_4w_c": prev_slope_c,
            "time_c": time_c,

            # step-known controls (seasonality varies with week; latitude constant)
            "latitude": float(wkrow["latitude"]),
            "sine_seasonality": float(wkrow["sine_seasonality"]),
            "cosine_seasonality": float(wkrow["cosine_seasonality"]),
            "sine_seasonality_x_latitude": float(wkrow["sine_seasonality_x_latitude"]),
            "cosine_seasonality_x_latitude": float(wkrow["cosine_seasonality_x_latitude"]),
        }

        # ------------------ (A) GATE: add lag_gate raw (will be z-scored) ------------------
        sr["lag_gate"] = 0.0  # serve with gate OFF

        # ------------------ (B) policy Koyck windows ------------------
        for name in policy_base_names:
            sr[name] = float(windows[name][h])

        # ------------------ (C) discrete lag features (RAW) ------------------
        for p in POLICY_COLS:
            for k in LAG_STEPS:
                lname = f"{p}_lag{k}"
                sr[lname] = float(lags[lname][h]) if lname in lags else feature_norm[lname]["mean"]

        # --- Z-score everything present ---
        z = {k: _normalize_scalar(k, v, feature_norm) for k, v in sr.items() if k in feature_norm}

        # ------------------ (D) GATE in Z-space: zero all lag features ------------------
        for p in POLICY_COLS:
            for k in LAG_STEPS:
                lname = f"{p}_lag{k}"
                if lname in z:
                    z[lname] = 0.0  # hard zero in Z-space when gate=0

        # (Interactions are OFF; keep no-op for compatibility)
        _recompute_interactions_inplace(sr, feature_norm, policy_base_names)

        # Ensure any (future) normalized interaction features present (no-ops here)
        for k in feature_order:
            if "_x_" in k and k not in z and k in sr:
                z[k] = float(sr[k])

        # Order to match training
        step_rows.append([z.get(n, 0.0) for n in feature_order])

    # [B=1, H, D]
    decoder_cont = torch.tensor(step_rows, dtype=torch.float32, device=device).unsqueeze(0)

    # Scripted model returns already scaled to [0, U] (U=1.0 per export unless changed)
    with torch.no_grad():
        y = model(decoder_cont).squeeze(0).cpu().numpy()  # [H, 3]

    weeks = [int(week_id) + h for h in range(1, H + 1)]
    out = pd.DataFrame({
        "country_iso3": country, "week_id": weeks,
        "q10": y[:, 0], "q50": y[:, 1], "q90": y[:, 2]
    })

    # === Δ over horizon relative to first step (h=1) ===
    base_q = out.loc[out.index.min(), ["q10","q50","q90"]].to_numpy(dtype=float)
    deltas = out[["q10","q50","q90"]].to_numpy(dtype=float) - base_q
    out["dq10"] = deltas[:, 0]
    out["dq50"] = deltas[:, 1]
    out["dq90"] = deltas[:, 2]

    return out

# ------------------- Convenience: inspect a country's week indices -------------------
def describe_country_weeks(country_iso3, n_head=5, n_tail=5):
    g = df[df["country_iso3"] == str(country_iso3)].sort_values("week_id").reset_index(drop=True)
    print(f"{country_iso3}: {len(g)} rows (positional indices 0..{len(g)-1})")
    try:
        from IPython.display import display
        display(g.head(n_head)[["week_id", "year", "iso_week"]])
        display(g.tail(n_tail)[["week_id", "year", "iso_week"]])
    except Exception:
        print(g.head(n_head)[["week_id", "year", "iso_week"]])
        print(g.tail(n_tail)[["week_id", "year", "iso_week"]])

# ------------------- Scenario comparison helper -------------------
def compare_horizons(country_iso3, week_id_or_idx, scenario_sliders):
    """
    Return only q50_baseline, q50_scenario, pct_change_q50.
    Baseline = observed policy path; Scenario = user-specified sliders.
    """
    country = str(country_iso3)
    g = df[df["country_iso3"] == country].sort_values("week_id").reset_index(drop=True)
    if g.empty:
        raise ValueError(f"No data for {country}")
    week_id = _resolve_week_id_for_country(g, week_id_or_idx)

    # Observed baseline policy path (t0+1..t0+H)
    baseline_path = _future_slider_path_from_data(g, week_id, H)

    preds_baseline = predict_horizon(country_iso3, week_id, baseline_path)
    preds_scenario = predict_horizon(country_iso3, week_id, scenario_sliders)

    df_cmp = preds_baseline.merge(
        preds_scenario, on=["country_iso3","week_id"], suffixes=("_baseline", "_scenario")
    )

    # % change in the q50 level (scenario vs baseline)
    denom = df_cmp["q50_baseline"].replace({0.0: np.nan})
    df_cmp["pct_change_q50"] = 100.0 * (df_cmp["q50_scenario"] - df_cmp["q50_baseline"]) / denom

    # Return only the requested columns (keep identifiers for clarity)
    return df_cmp[["country_iso3", "week_id", "q50_baseline", "q50_scenario", "pct_change_q50"]]


# ------------------- Example usage -------------------
# describe_country_weeks("USA")
# preds = predict_horizon("USA", 120, (40.0, 50.0, 60.0))   # raw levels
# from IPython.display import display
# display(preds.head(12))   # includes dq10/dq50/dq90 relative to h=1
#
# cmp = compare_horizons("AUT", 91, (0.0, 0.0, 0.0))
# display(cmp.head(12))     # includes pct_change_q50


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded artifacts from: /content/drive/MyDrive/Models


In [8]:
# ------------------- Example usage -------------------
# describe_country_weeks("USA")
#preds = predict_horizon("USA", 120, (40.0, 50.0, 60.0))   # raw levels
#from IPython.display import display
#display(preds.head(12))
#
cmp = compare_horizons("BRA", 10, (0.9, 0.9, 0.9))
display(cmp.head(12))

,country_iso3,week_id,q50_baseline,q50_scenario,pct_change_q50
0,BRA,11,0.167207,0.161828,-3.216837
1,BRA,12,0.165357,0.155102,-6.201385
2,BRA,13,0.160382,0.146096,-8.907748
3,BRA,14,0.134400,0.126744,-5.696643
4,BRA,15,0.133707,0.124561,-6.840617
5,BRA,16,0.133149,0.122612,-7.913980
6,BRA,17,0.132846,0.121048,-8.881175
7,BRA,18,0.131807,0.118943,-9.759065
8,BRA,19,0.129922,0.116203,-10.559095
9,BRA,20,0.128909,0.114367,-11.280685
